# Bayes' Theorem: Is This Message Spam?

This notebook uses a tiny, made-up inbox to answer one question:

> If a message contains the word **prize**, how likely is it to be spam?

We will first **count real cases**, then use Bayes' theorem to get the same answer with a formula.

Run each cell from top to bottom. The most useful learning happens when you change the numbers in the final section and run it again.

## 1. Start with a concrete inbox

Imagine we inspect **1,000 messages**:

| Message type | Total messages | Contains “prize” |
|---|---:|---:|
| Spam | 100 | 80 |
| Not spam | 900 | 90 |

There are two different questions we could ask:

1. Among spam messages, how often does “prize” appear?  
   This is `P(prize | spam)`.
2. Among messages containing “prize”, how often is the message spam?  
   This is `P(spam | prize)`.

These are **not** the same question. Bayes' theorem helps us turn the first kind of information into the second.

In [14]:
# These are counts, not probabilities yet.
total_spam = 100
total_not_spam = 900
prize_in_spam = 80
prize_in_not_spam = 90

total_messages = total_spam + total_not_spam
total_prize_messages = prize_in_spam + prize_in_not_spam

print(f"Total messages: {total_messages}")
print(f'Messages containing "prize": {total_prize_messages}')

Total messages: 1000
Messages containing "prize": 170


## 2. Answer the question by counting

Look only at the messages that contain “prize”. There are 170 of them:

- 80 are spam
- 90 are not spam

So the answer is simply:

`P(spam | prize) = 80 / 170 = 0.4706`, or about **47%**.

The vertical bar means “given that”. We are asking for the probability of spam **given that** the word “prize” appeared.

In [15]:
# Count the spam messages inside the group that contains "prize".
prize_in_spam = 80
total_prize_messages = 170
probability_spam_given_prize_by_counting = prize_in_spam / total_prize_messages

print(f"P(spam | prize) = {probability_spam_given_prize_by_counting:.4f}")
print(f"That is about {probability_spam_given_prize_by_counting:.0%}.")

P(spam | prize) = 0.4706
That is about 47%.


## 3. Notice the common mistake

It is tempting to say “80%”, because 80 out of 100 spam messages contain “prize”.

But that number answers a different question:

`P(prize | spam) = 80 / 100 = 80%`

The spam filter needs `P(spam | prize)`, not `P(prize | spam)`. Bayes' theorem is the bridge between them.

In [16]:
prize_in_spam = 80
total_spam = 100
probability_prize_given_spam = prize_in_spam / total_spam

print(f"P(prize | spam) = {probability_prize_given_spam:.0%}")
print(f"P(spam | prize) = {probability_spam_given_prize_by_counting:.0%}")

P(prize | spam) = 80%
P(spam | prize) = 47%


## 4. Use Bayes' theorem

Bayes' theorem says:

`P(spam | prize) = P(prize | spam) × P(spam) / P(prize)`

Each part has a plain-English meaning:

- `P(spam)`: the **prior** — how common spam is before seeing the word.
- `P(prize | spam)`: the **likelihood** — how often spam contains the word.
- `P(prize)`: the **evidence** — how common the word is in all messages.
- `P(spam | prize)`: the **posterior** — our updated belief after seeing the word.

In [17]:
prior_spam = total_spam / total_messages
likelihood_prize_given_spam = prize_in_spam / total_spam
evidence_prize = total_prize_messages / total_messages

posterior_spam_given_prize = (
    likelihood_prize_given_spam * prior_spam / evidence_prize
)

print(f"Prior P(spam): {prior_spam:.2%}")
print(f"Likelihood P(prize | spam): {likelihood_prize_given_spam:.2%}")
print(f"Evidence P(prize): {evidence_prize:.2%}")
print(f"Posterior P(spam | prize): {posterior_spam_given_prize:.2%}")

Prior P(spam): 10.00%
Likelihood P(prize | spam): 80.00%
Evidence P(prize): 17.00%
Posterior P(spam | prize): 47.06%


### Why does the formula agree with counting?

The numerator finds the messages that are both spam **and** contain “prize”.
The denominator expands the group to include every message containing “prize”.

This is exactly the same group-counting calculation we did above, just written using probabilities.

In [18]:
# Check that both methods give the same answer.
difference = abs(
    posterior_spam_given_prize - probability_spam_given_prize_by_counting
)
print(f"Difference between the two answers: {difference:.10f}")
assert difference < 1e-12

Difference between the two answers: 0.0000000000


## 5. The surprising part: the base rate matters

In our example, 10% of all messages were spam. That is the prior.

The function below keeps the word behavior the same but lets us change how common spam is. This shows why a clue can be strong without making the conclusion certain.

In [19]:
def probability_of_spam_given_prize(
    spam_rate, prize_rate_in_spam, prize_rate_in_not_spam
):
    "Return P(spam | prize) using Bayes' theorem."
    probability_prize = (
        prize_rate_in_spam * spam_rate
        + prize_rate_in_not_spam * (1 - spam_rate)
    )
    return prize_rate_in_spam * spam_rate / probability_prize


prize_rate_in_spam = 0.80
prize_rate_in_not_spam = 0.10

for spam_rate in [0.01, 0.10, 0.50]:
    posterior = probability_of_spam_given_prize(
        spam_rate, prize_rate_in_spam, prize_rate_in_not_spam
    )
    print(f"If spam is {spam_rate:.0%} of messages, P(spam | prize) = {posterior:.0%}")

If spam is 1% of messages, P(spam | prize) = 7%
If spam is 10% of messages, P(spam | prize) = 47%
If spam is 50% of messages, P(spam | prize) = 89%


## 6. Try it yourself

Change one number at a time in the next cell, then run it again:

- Make the word appear in more normal messages.
- Make spam more common or more rare.
- Make the word appear in fewer spam messages.

Before running the cell, predict whether the posterior probability will go up or down. Then compare your prediction with the result.

In [20]:
# Edit these three values and run this cell again.
spam_rate = 0.10
prize_rate_in_spam = 0.80
prize_rate_in_not_spam = 0.10

answer = probability_of_spam_given_prize(
    spam_rate, prize_rate_in_spam, prize_rate_in_not_spam
)
print(f"P(spam | prize) = {answer:.2%}")
print(f"P(prize | spam) = {prize_rate_in_spam:.2%}")

P(spam | prize) = 47.06%
P(prize | spam) = 80.00%


## Takeaway

Bayes' theorem updates a starting belief using new evidence:

`posterior = likelihood × prior / evidence`

The most important habit is to read the direction of the conditional probability carefully:

- `P(prize | spam)` asks: “If it is spam, how likely is the word?”
- `P(spam | prize)` asks: “If I see the word, how likely is spam?”

They answer different questions because the **base rate** of spam also matters.